In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
# Raw Data에서 다시 시작
df = pd.read_csv("../01_raw/uci-secom.csv")

# 공정 측정 Feature 0 ~ 589
sensor_cols = [str(i) for i in range(590)]

# X: 공정 측정 변수
X = df[sensor_cols].copy()

# y: Pass = 0, Fail = 1
y = (df["Pass/Fail"] == 1).astype(int)

print("X Shape:", X.shape)
print("y Shape:", y.shape)

print()
print("Target Distribution:")
print(y.value_counts())

X Shape: (1567, 590)
y Shape: (1567,)

Target Distribution:
Pass/Fail
0    1463
1     104
Name: count, dtype: int64


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
    )

In [5]:
print("Train Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

print()
print("Train Target Counts:")
print(y_train.value_counts())

print()
print("Test Target Counts:")
print(y_test.value_counts())

print()
print("Train Fail Rate:")
print(y_train.mean() * 100)

print()
print("Test Fail Rate:")
print(y_test.mean() * 100)

Train Shape: (1253, 590)
Test Shape : (314, 590)

Train Target Counts:
Pass/Fail
0    1170
1      83
Name: count, dtype: int64

Test Target Counts:
Pass/Fail
0    293
1     21
Name: count, dtype: int64

Train Fail Rate:
6.624102154828412

Test Fail Rate:
6.687898089171974


In [6]:
# phase1의 정보를 절대 사용하지 않는다.(data leakage 방지), Train data내에서 정보를 다시 정립한다

# Train 데이터만 사용한 Feature Quality Audit

# 1. 모든 값이 NaN인 Feature
train_all_missing = [
    col for col in X_train.columns
    if X_train[col].isna().all()
]

# 2. NaN 없이 모든 값이 동일한 Strict Constant Feature
train_strict_constant = [
    col for col in X_train.columns
    if X_train[col].notna().all()
    and X_train[col].nunique() == 1
]

# 3. 관측값은 한 종류지만 일부 NaN이 존재하는 Feature
train_observed_constant_with_missing = [
    col for col in X_train.columns
    if X_train[col].dropna().nunique() == 1
    and X_train[col].isna().any()
]

print("Train All-Missing Features:")
print(train_all_missing)
print("Count:", len(train_all_missing))

print("\nTrain Strict Constant Features:")
print(train_strict_constant)
print("Count:", len(train_strict_constant))

print("\nTrain Observed-Constant + Missing Features:")
print(train_observed_constant_with_missing)
print("Count:", len(train_observed_constant_with_missing))

Train All-Missing Features:
[]
Count: 0

Train Strict Constant Features:
[]
Count: 0

Train Observed-Constant + Missing Features:
['5', '13', '42', '49', '52', '69', '97', '141', '149', '178', '179', '186', '189', '190', '191', '192', '193', '194', '226', '229', '230', '231', '232', '233', '234', '235', '236', '237', '240', '241', '242', '243', '256', '257', '258', '259', '260', '261', '262', '263', '264', '265', '266', '276', '284', '313', '314', '315', '322', '325', '326', '327', '328', '329', '330', '364', '369', '370', '371', '372', '373', '374', '375', '378', '379', '380', '381', '394', '395', '396', '397', '398', '399', '400', '401', '402', '403', '404', '414', '422', '449', '450', '451', '458', '461', '462', '463', '464', '465', '466', '481', '498', '501', '502', '503', '504', '505', '506', '507', '508', '509', '512', '513', '514', '515', '528', '529', '530', '531', '532', '533', '534', '535', '536', '537', '538']
Count: 116


In [7]:
# Train 기준 Feature별 Missing Rate
train_missing_rate = X_train.isna().mean() * 100

print("Maximum Train Missing Rate:")
print(train_missing_rate.max())

print("\nFeatures with Missing Rate >= 50%:")
print((train_missing_rate >= 50).sum())

print("\nFeatures with Missing Rate >= 80%:")
print((train_missing_rate >= 80).sum())

print("\nFeatures with Missing Rate >= 90%:")
print((train_missing_rate >= 90).sum())

Maximum Train Missing Rate:
90.66241021548284

Features with Missing Rate >= 50%:
24

Features with Missing Rate >= 80%:
8

Features with Missing Rate >= 90%:
4


In [8]:
# 결측률 높은 순서 확인
train_missing_summary = (
    train_missing_rate
    .sort_values(ascending=False)
    .head(20)
)

display(train_missing_summary)

292    90.662410
293    90.662410
158    90.662410
157    90.662410
220    85.634477
85     85.634477
358    85.634477
492    85.634477
384    66.320830
382    66.320830
383    66.320830
245    66.320830
109    66.320830
244    66.320830
516    66.320830
246    66.320830
110    66.320830
517    66.320830
111    66.320830
518    66.320830
dtype: float64